<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c24-header.png" alt="Nextia Learning · AI Security, Privacy and Responsible Design" width="100%">

# Find and remove personal data before it reaches a model

**[Find and remove personal data before it reaches a model](https://learning.nextia-ai.com/courses/ai-security/m07/find-and-remove-personal-data/)** · AI Security, Privacy and Responsible Design · Module 7, case study 1 of 2 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** find and mask personal data in financial documents in English, French and German, before the text goes to a language model. You compare three detectors: simple rules, Microsoft Presidio running on your computer, and a recorded language model. You measure what each one **misses** and what it **masks too much**, by type of value and by language. Then you decide what still needs a person, and what the masking costs the model.

| | |
|---|---|
| **Time** | About 2 hours with the lesson page. The notebook runs in about 3 minutes (most of it is the setup). |
| **Needs** | An internet connection for the setup cells. No account, no key, no graphics card: the model's answers are real recordings, replayed. |
| **Recorded** | gpt-6-luna as `chat-small` on Azure: 2,700 answers in 3 recordings. |
| **You should know** | [Data inventory](https://learning.nextia-ai.com/courses/ai-security/m04/data-inventory/), [Minimize and retain deliberately](https://learning.nextia-ai.com/courses/ai-security/m04/minimize-and-retain-deliberately/) and [Bias and accessibility](https://learning.nextia-ai.com/courses/ai-security/m05/bias-and-accessibility/) in this course. |
| **Data** | Gretel's synthetic_pii_finance_multilingual (Apache-2.0), test files in English, French and German: a fixed sample of 1,320 documents. |
| **Tested** | Python 3.12 with presidio-analyzer 2.2.364, spaCy 3.8.16 and pandas 3.0.6, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

> **Warning.** Every person, account and number in this data is made up. Use the same method on real data only on a computer and in a place that may hold that data.


### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ai-security/m07/find-and-remove-personal-data/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C24/M07-L01-find-and-remove-personal-data/find-and-remove-personal-data-solution.ipynb).

## Setup: the packages and the language models

Run the next cell. It installs Microsoft Presidio 2.2.364 (MIT licence), spaCy 3.8.16, and three small spaCy models (MIT licence): English, German, and a multilingual model that the notebook uses for French. The models are about 40 MB in all. The cell checks the SHA-256 of each model file. On Colab this takes one to two minutes. You should see `presidio-analyzer 2.2.364 | spaCy 3.8.16 | models: en_core_web_sm, de_core_news_sm, xx_ent_wiki_sm`.

If the cell shows an error, read its last line. `Could not install`: check your internet connection (in Kaggle, turn on **Internet** in the settings). `wrong checksum` (pip says `THESE PACKAGES DO NOT MATCH THE HASHES`): the file changed at its source; stop and tell us. If you see other versions, restart the session (in Colab: **Runtime › Restart session**) and run the cell again.

In [ ]:
import importlib, os, subprocess, sys
MODELS = {
    "en_core_web_sm": "1932429db727d4bff3deed6b34cfc05df17794f4a52eeb26cf8928f7c1a0fb85",
    "de_core_news_sm": "fec69fec52b1780f2d269d5af7582a5e28028738bd3190532459aeb473bfa3e7",
    "xx_ent_wiki_sm": "6f3c4b853852ea9e9d2dc76cc950dddb10a7e4c42d813308caefe6c5e8be2f0a"
}
URL = "https://github.com/explosion/spacy-models/releases/download/{0}-3.8.0/{0}-3.8.0-py3-none-any.whl#sha256={1}"
need = ["presidio-analyzer==2.2.364", "spacy==3.8.16", "pandas", "pyarrow", "matplotlib"]
need += [URL.format(m, sha) for m, sha in MODELS.items() if importlib.util.find_spec(m) is None]
if subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need]).returncode:
    raise SystemExit("Could not install the packages. Check your internet connection.")
importlib.invalidate_caches()
from importlib.metadata import version
print("presidio-analyzer", version("presidio-analyzer"), "| spaCy", version("spacy"), "| models:", ", ".join(MODELS))

## Setup: the documents and the recorded model answers

Run the next cell. It makes a folder `pii-data` and downloads two things, and checks the SHA-256 of every file:

1. The test files of Gretel's *synthetic_pii_finance_multilingual* dataset in English, French and German (about 3.5 MB), from a fixed version on Hugging Face. The dataset is under the Apache-2.0 licence. Every person, account and address in it is made up.
2. The recorded answers of the model `chat-small` (gpt-6-luna on Azure) from the course's labs repository (about 0.3 MB). You need no account and no key.

You should see `Ready: 3 document files, 3 recordings.`

In [ ]:
import gzip, hashlib, json, urllib.request
from pathlib import Path
HF = "https://huggingface.co/datasets/gretelai/synthetic_pii_finance_multilingual/resolve/7b844d16738527a04264f50214cb426a4cea0897/data/"
LABS = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C24/M07-L01-find-and-remove-personal-data/")
DOCS = {
    "en": [
        "English_test-00000-of-00001.parquet",
        "c02b06d3c5b7c375525136d6f74acc52ab8fc07fa863d0aa50734910ec0ef2ad"
    ],
    "fr": [
        "France_test-00000-of-00001.parquet",
        "39fc2777822bbecab2d89bb23cbed40d9825ff4117897b8b025c02287f4ba845"
    ],
    "de": [
        "German_test-00000-of-00001.parquet",
        "bfc64380bc24453f3460ffee6afbd7a3442b60dd26d5341c9fe0f1cca43216c3"
    ]
}
RECS = {
    "recordings/chat-small__detect.jsonl.gz": "5ddb002be7b2b37bc482390df503a3a6e57a98cae90631bdbde2a1116e9f5b2d",
    "recordings/chat-small__doctype.jsonl.gz": "104c2dada9f50b73e2b9bfe35ae916d25bd0ab9d78acf8362e6fdbe5f26294db",
    "recordings/chat-small__facts.jsonl.gz": "94c130916433c099140a069a0985be63013cf028c828db6da598b24d5897c9aa"
}
WORK = Path("pii-data").resolve()
(WORK / "recordings").mkdir(parents=True, exist_ok=True)

def fetch(url, path, expected):
    """Download url to path unless a file with the right SHA-256 is already there."""
    if path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected:
        return
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {path.name}. Check your internet connection.")
    if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise SystemExit(f"{path.name} has the wrong checksum. Delete the folder pii-data and run again.")

for lang, (name, sha) in DOCS.items():
    fetch(HF + name, WORK / name, sha)
for name, sha in RECS.items():
    fetch(LABS + name, WORK / name, sha)
recordings = {}
for name in RECS:
    task = name.split("__")[1][: -len(".jsonl.gz")]
    recordings[task] = [json.loads(line) for line in gzip.decompress((WORK / name).read_bytes()).decode().splitlines()]
print(f"Ready: {len(DOCS)} document files, {len(recordings)} recordings.")

## Setup: helpers

The next cell defines the check helpers, `expect` and `expect_hash`, and `wilson(k, n)`, the 95% interval for k out of n from the evaluation course's [Uncertainty and slices](https://learning.nextia-ai.com/courses/evals/m05/uncertainty-and-slices/). Run it. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def wilson(k, n, z=1.96):
    """95% Wilson interval for k out of n, as (low, high)."""
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * ((p * (1 - p) / n + z * z / (4 * n * n)) ** 0.5) / (1 + z * z / n)
    return centre - half, centre + half

def pct(x):
    return f"{100 * x:.1f}%"

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The masking policy

Before you look for personal data, write down what counts. The next cell holds this case study's policy:

- `MUST_MASK`: labels of the dataset that are personal data and must not reach the model, grouped for the report (11 groups).
- `KEEP`: labels that the model needs for its work: company names, ordinary dates and times, and bank codes (SWIFT/BIC). Masking them protects nobody and costs meaning.
- A **placeholder** is a template slot such as `[Date]` or `MM/DD/YYYY`. It holds no personal data, so it counts neither way.

Run it. It prints nothing.

In [ ]:
import json
import re
from collections import Counter, defaultdict

import pandas as pd

SEED = 24
REVISION = "7b844d16738527a04264f50214cb426a4cea0897"   # the dataset's commit on Hugging Face
FILES = {   # language code -> (file in the dataset, SHA-256)
    "en": ("English_test-00000-of-00001.parquet", "c02b06d3c5b7c375525136d6f74acc52ab8fc07fa863d0aa50734910ec0ef2ad"),
    "fr": ("France_test-00000-of-00001.parquet", "39fc2777822bbecab2d89bb23cbed40d9825ff4117897b8b025c02287f4ba845"),
    "de": ("German_test-00000-of-00001.parquet", "bfc64380bc24453f3460ffee6afbd7a3442b60dd26d5341c9fe0f1cca43216c3"),
}
LANGUAGES = {"en": "English", "fr": "French", "de": "German"}
DEV_PER_LANGUAGE, TEST_PER_LANGUAGE = 140, 300

# The masking policy: which labels are personal data that must not reach the model (grouped for the
# report), and which labels the model needs to keep (masking them costs meaning, not privacy).
MUST_MASK = {
    "name": "name", "first_name": "name", "last_name": "name",
    "email": "email", "phone_number": "phone",
    "street_address": "address", "local_latlng": "address",
    "ssn": "government ID", "passport_number": "government ID", "driver_license_number": "government ID",
    "customer_id": "customer or staff ID", "employee_id": "customer or staff ID",
    "iban": "bank account", "bban": "bank account", "bank_routing_number": "bank account",
    "credit_card_number": "card", "credit_card_security_code": "card",
    "password": "secret", "api_key": "secret", "account_pin": "secret",
    "ipv4": "online ID", "ipv6": "online ID", "user_name": "online ID",
    "date_of_birth": "date of birth",
}
KEEP = {"company": "company", "date": "date or time", "time": "date or time", "date_time": "date or time",
        "swift_bic_code": "bank code"}
GROUPS = ["name", "address", "phone", "email", "customer or staff ID", "bank account", "government ID",
          "card", "secret", "online ID", "date of birth"]
PLACEHOLDER = re.compile(r"[\[\]{}<>]|^[XxMDY#*_ /.\-]+$")


def is_placeholder(value):
    """A template slot such as '[Date]' or 'MM/DD/YYYY': it holds no personal data."""
    return bool(PLACEHOLDER.search(value))

## 2. The data, briefly

The next cell reads the three files and takes a fixed random sample of 440 documents per language. The first 140 of each language are the **development part**: you may look at them, write rules on them and choose settings. The other 300 are the **test part**: you score each design on it once, at the end. Then it lists every labelled value with its role in the policy.

> **Check.** You should see 1,320 documents, 420 in the development part and 900 in the test part, and a table of labelled values by role.

In [ ]:
def load_sample(paths):
    """Read the three test files, fix the sample, and split it into a development part and a test part.
    `paths` maps a language code to a local parquet file. Returns one row per document."""
    parts = []
    for lang, path in paths.items():
        df = pd.read_parquet(path)[["document_type", "generated_text", "pii_spans"]]
        df = df.sample(n=DEV_PER_LANGUAGE + TEST_PER_LANGUAGE, random_state=SEED).reset_index(drop=True)
        df["split"] = ["dev"] * DEV_PER_LANGUAGE + ["test"] * TEST_PER_LANGUAGE
        df["lang"] = lang
        df["doc_id"] = [f"{lang}-{i:03d}" for i in range(len(df))]
        parts.append(df)
    docs = pd.concat(parts, ignore_index=True).rename(columns={"generated_text": "text"})
    docs["spans"] = docs.pii_spans.map(json.loads)
    return docs.drop(columns="pii_spans")[["doc_id", "lang", "split", "document_type", "text", "spans"]]


def gold_values(docs):
    """Every labelled value, one row each, with its role in the policy: mask, keep or placeholder."""
    rows = []
    for d in docs.itertuples():
        for s in d.spans:
            value = d.text[s["start"]:s["end"]]
            label = s["label"]
            role = ("placeholder" if is_placeholder(value) else "mask" if label in MUST_MASK
                    else "keep" if label in KEEP else "other")
            rows.append({"doc_id": d.doc_id, "lang": d.lang, "split": d.split, "label": label,
                         "group": MUST_MASK.get(label) or KEEP.get(label), "role": role,
                         "start": s["start"], "end": s["end"], "value": value})
    return pd.DataFrame(rows)

docs = load_sample({lang: WORK / name for lang, (name, _) in DOCS.items()})
gold = gold_values(docs)
dev, test = docs[docs.split == "dev"], docs[docs.split == "test"]
print(len(docs), "documents |", docs.split.value_counts().to_dict())
gold.groupby(["split", "role"]).size().unstack()

Look at one document with its labels. The next cell prints a document with each labelled value marked as `[[label: value]]`.

> **Predict.** In a bank's letter, which kinds of personal data do you expect most often? Write two kinds, then run the cell and the one after it.

In [ ]:
def show_labels(doc_id, width=900):
    """Print the start of a document with each labelled value marked as [[label: value]]."""
    d = docs.set_index("doc_id").loc[doc_id]
    out, last = [], 0
    for s in sorted(d.spans, key=lambda s: s["start"]):
        if s["start"] < last:
            continue
        out += [d.text[last:s["start"]], f"[[{s['label']}: {d.text[s['start']:s['end']]}]]"]
        last = s["end"]
    out.append(d.text[last:])
    print(f"{doc_id} | {d.lang} | {d.document_type}")
    print("".join(out)[:width])

show_labels("de-005")

In [ ]:
gold[(gold.role == "mask") & (gold.split == "test")].groupby(["group", "lang"]).size().unstack().loc[GROUPS]

> **Check.** Names and addresses are by far the most common personal values. Card numbers, secrets and dates of birth are rare: in the test part, some groups have fewer than 20 values in a language. Remember that when you read a percentage for them.

## 3. Check the labels before you trust them

A labelled dataset is someone's work, and work has mistakes. Your scores can only be as good as the labels. The dataset's card says that its labels came from a named-entity model and a language-model judge, and that "some errors may still be present".

The next cell prints the placeholders in the development part: values that the dataset labels, but that hold no personal data.

In [ ]:
gold[(gold.split == "dev") & (gold.role == "placeholder")].value.value_counts().head(10)

> **Your turn.** Count the **email-shaped strings** in the development documents that have **no label at all**. Use the pattern `EMAIL` below, and count a match as labelled when it overlaps any labelled span of its document. Put the number in `unlabelled_emails`.

In [ ]:
import re
EMAIL = re.compile(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+")
unlabelled_emails = ...  # your answer: a number

In [ ]:
expect('email-shaped strings with no label in the development part', unlabelled_emails, 52)

> **Check.** Many addresses have no label. Some are a team's address (`support@…`), but many name a person (`jean.dupont@example.com`). So a detector that masks them is **right**, but the score counts it as over-masking. And a personal value with no label that a detector misses is not counted as a miss at all. Keep both in mind: the measured misses are a **lower bound**.

## 4. The idea on a tiny example

One sentence, five labelled values, one detector. The next cell runs the **rules** detector (it is defined in the cell after the tiny example, so this cell defines it first) and scores each labelled value:

- **hidden**: at least 80% of its letters and digits are masked;
- **partly**: some are masked, but less than 80%;
- **missed**: none is masked.

The date is labelled, but the policy keeps dates, so it is not scored as personal data.

> **Predict.** Rules match shapes: an email, a card number, a phone number. Which of the four personal values will they hide? Write your answer, then run the two cells.

In [ ]:
# Detector (a): rules. Each rule is a pattern for one shape of value, or a value after a keyword.
# Written and changed on the development part only (see the experiment log).
CONTEXT = {  # keywords in English, French and German, then the value that follows them
    "SECRET": r"(?:password|passwort|kennwort|mot de passe|pin|cvv|cvc|security code|sicherheitscode|cryptogramme)",
    "ADDRESS": r"(?:address|adresse|anschrift|domicilié à|wohnhaft in)",
    "BIRTH": r"(?:date of birth|dob|born on|born|geburtsdatum|geboren am|geboren|date de naissance|né le|née le|né\(e\) le)",
}
RULES = [
    ("EMAIL", r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+"),
    ("IBAN", r"\b[A-Z]{2}\d{2}(?: ?[A-Z0-9]{4}){2,7}(?: ?[A-Z0-9]{1,3})?\b"),
    ("CARD", r"\b\d{4}[ -]?\d{4}[ -]?\d{4}[ -]?\d{3,4}\b"),
    ("IP", r"\b(?:\d{1,3}\.){3}\d{1,3}\b|\b(?:[0-9a-fA-F]{1,4}:){7}[0-9a-fA-F]{1,4}\b"),
    ("SSN", r"\b\d{3}-\d{2}-\d{4}\b"),
    ("PHONE", r"(?<![\w.:/-])(?:\+\d{1,3}[ .-]?)?(?:\(\d{1,4}\)[ .-]?)?\d[\d .-]{6,14}\d(?![\w:/-]|\.\d)"),
    ("NUMBER", r"\b\d{9,20}\b"),
    ("ID", r"\b(?!(?:USD|EUR|GBP|CHF|SEK|ISO|REF|INV)\d)[A-Z][A-Za-z]{0,3}-?\d{5,9}(?:-[A-Za-z]{1,2})?\b"),
    ("KEY", r"\b(?=[\w-]*\d)(?=[\w-]*[a-z])(?=[\w-]*[A-Z])[\w-]{20,}\b"),
    ("SECRET", CONTEXT["SECRET"] + r"\s*[:=]\s*(\S+)"),
    ("ADDRESS", CONTEXT["ADDRESS"] + r"\s*:\s*([^\n<>\[\]]{6,80})"),
    ("STREET", r"\b\d{1,5}[a-z]?,? (?:[A-ZÀ-Ý][\w'’.-]+ ){0,3}(?:rue|avenue|av\.|boulevard|bd|place|chemin|allée|quai|"
               r"impasse|via|calle|plaza|rambla)\b[^\n,]{0,40}"
               r"|\b(?:\d{1,5} )?[A-ZÀ-Ý][\w'’-]*(?:straße|strasse|str\.|weg|gasse|platz|allee|ring|damm|"
               r"straat|laan|gatan|vägen|gränd)(?: \d{1,5}[a-z]?)?\b"
               r"|\b\d{1,5} (?:[A-Z][a-z]+ ){1,3}(?:Street|St\.|Avenue|Ave\.?|Road|Rd\.|Lane|Drive|Way|Court|Square|"
               r"Place|Boulevard|Blvd\.?)(?:,? (?:Apt\.|Suite) ?\w+)?"),
    ("BIRTH", CONTEXT["BIRTH"] + r"\s*[:,]?\s*(\d{1,2}[./-]\d{1,2}[./-]\d{2,4}|\d{4}-\d{2}-\d{2}"
              r"|\d{1,2}\.? \w+ \d{4}|\w+ \d{1,2}, \d{4}|1er \w+ \d{4})"),
]
RULES = [(label, re.compile(pattern, re.IGNORECASE if label in ("SECRET", "BIRTH", "ADDRESS") else 0)) for label, pattern in RULES]
DATE_SHAPE = re.compile(r"^\d{1,4}[./-]\d{1,2}[./-]\d{1,4}$|^(?:19|20)\d{2}[ -]")


def detect_rules(text, lang=None):
    """Spans (start, end, label) that match one of the rules. For a keyword rule, only the value is masked.
    A phone match must have 9 to 15 digits and must not look like a date."""
    spans = []
    for label, pattern in RULES:
        for m in pattern.finditer(text):
            start, end = (m.start(1), m.end(1)) if pattern.groups else (m.start(), m.end())
            value = text[start:end]
            if label == "PHONE":
                digits = sum(ch.isdigit() for ch in value)
                if not 9 <= digits <= 15 or DATE_SHAPE.match(value):
                    continue
            spans.append((start, end, label))
    return spans

def merge(spans):
    """Join overlapping spans. The label comes from the span that starts first; on a tie, the longer one, then
    the one listed first (rules are listed before Presidio, and in the order of RULES)."""
    out = []
    for s, e, label in sorted(spans, key=lambda x: (x[0], -x[1])):
        if out and s <= out[-1][1]:
            out[-1] = (out[-1][0], max(out[-1][1], e), out[-1][2])
        else:
            out.append((s, e, label))
    return out


def mask(text, spans):
    """Replace each span with a typed marker, such as <EMAIL>, and keep the rest of the text."""
    out, last = [], 0
    for s, e, label in merge(spans):
        out.append(text[last:s])
        out.append(f"<{label}>")
        last = e
    out.append(text[last:])
    return "".join(out)

HIDDEN_SHARE = 0.8   # a value counts as hidden when at least 80% of its letters and digits are masked


def masked_chars(text, spans):
    """The set of positions that a detector masks."""
    hit = set()
    for s, e, _ in spans:
        hit.update(range(s, e))
    return hit


def alnum_positions(text, start, end):
    return [i for i in range(start, end) if text[i].isalnum()]


def score_values(docs, gold, detections):
    """For every labelled value: the share of its letters and digits that the detector masked, and
    'hidden', 'partly' or 'missed'. `detections` maps doc_id -> spans."""
    texts = dict(zip(docs.doc_id, docs.text))
    hits = {d: masked_chars(texts[d], detections.get(d, [])) for d in texts}
    shares = []
    for g in gold.itertuples():
        pos = alnum_positions(texts[g.doc_id], g.start, g.end)
        shares.append(sum(p in hits[g.doc_id] for p in pos) / len(pos) if pos else 1.0)
    out = gold.copy()
    out["masked_share"] = shares
    out["status"] = ["hidden" if x >= HIDDEN_SHARE else "partly" if x > 0 else "missed" for x in shares]
    return out


def over_masking(docs, gold, detections):
    """Letters and digits masked outside every personal value, per document, split by where they were:
    in a value the model needs (company, date, bank code) or in other text."""
    rows = []
    for d in docs.itertuples():
        g = gold[gold.doc_id == d.doc_id]
        personal, keep = set(), set()
        for v in g.itertuples():
            target = personal if v.role in ("mask", "placeholder") else keep
            target.update(range(v.start, v.end))
        hit = masked_chars(d.text, detections.get(d.doc_id, []))
        alnum = {i for i, ch in enumerate(d.text) if ch.isalnum()}
        outside = alnum - personal
        over = hit & outside
        rows.append({"doc_id": d.doc_id, "lang": d.lang, "chars_outside": len(outside),
                     "over_in_keep": len(over & keep), "over_in_other": len(over - keep)})
    return pd.DataFrame(rows)


def summary(docs, gold, detections):
    """The two headline numbers on one part of the data: share of personal values hidden, and share of
    the other letters and digits masked (over-masking)."""
    scored = score_values(docs, gold[gold.role == "mask"], detections)
    over = over_masking(docs, gold, detections)
    return {"values": len(scored),
            "hidden": round((scored.status == "hidden").mean(), 4),
            "partly": round((scored.status == "partly").mean(), 4),
            "missed": round((scored.status == "missed").mean(), 4),
            "over_masked": round((over.over_in_keep.sum() + over.over_in_other.sum()) / over.chars_outside.sum(), 4)}

In [ ]:
TINY = 'Dear Ms Ana Lima, your card 4111 1111 1111 1111 was blocked on 12/03/2024. Call Joe on +44 20 7946 0958.'
TINY_GOLD = [('Ana Lima', 'name'), ('4111 1111 1111 1111', 'credit_card_number'), ('12/03/2024', 'date'), ('Joe', 'first_name'), ('+44 20 7946 0958', 'phone_number')]
spans = []
for value, label in TINY_GOLD:
    start = TINY.index(value)
    spans.append({"start": start, "end": start + len(value), "label": label})
tiny_docs = pd.DataFrame([{"doc_id": "tiny", "lang": "en", "split": "tiny", "document_type": "Letter",
                           "text": TINY, "spans": spans}])
tiny_gold = gold_values(tiny_docs)
found_tiny = detect_rules(TINY)
print(mask(TINY, found_tiny))
score_values(tiny_docs, tiny_gold[tiny_gold.role == "mask"], {"tiny": found_tiny})[["value", "group", "masked_share", "status"]]

> **Check.** The rules hide the card number and the phone number: they have a clear shape. They miss both names: a name has no shape. This is the main limit of rules, and names are the most common personal value in the data.

> **Your turn.** Write the four statuses of the tiny example in order, as a list of strings, in `tiny_status`.

In [ ]:
tiny_status = ...  # your answer, for example ["hidden", "missed", ...]

In [ ]:
expect_hash('the statuses of the tiny example', tiny_status, 'ae017166a2d394f8')

## 5. Three detectors

**(a) Rules** you have already: patterns for emails, IBANs, card numbers, IP addresses, phone numbers, long numbers and IDs, and keyword rules ("password:", "date of birth", "address:"). They were written on the development part.

**(b) Presidio** finds names and places with a spaCy language model, and shapes with its own recognizers. It runs on your computer. The next cell defines it and runs it once on every document; it takes about half a minute. `presidio_defaults` keeps every finding; `presidio_tuned` drops dates, organisations and URLs, and findings with a score under 0.4.

> **Check.** You should see `Presidio ran on 1320 documents`.

In [ ]:
# Detector (b): Microsoft Presidio (MIT) with small spaCy models (MIT), on the CPU. French uses spaCy's
# multilingual model, because the French model's licence (LGPL-LR) is not on the course's list.
SPACY_MODELS = {"en": "en_core_web_sm", "fr": "xx_ent_wiki_sm", "de": "de_core_news_sm"}
NOT_PERSONAL = {"DATE_TIME", "NRP", "ORGANIZATION", "URL", "MEDICAL_LICENSE", "CRYPTO", "MAC_ADDRESS"}


def make_presidio():
    """One Presidio analyzer for the three languages, with every built-in recognizer that supports them."""
    from presidio_analyzer import AnalyzerEngine, RecognizerRegistry
    from presidio_analyzer.nlp_engine import NlpEngineProvider
    config = {"nlp_engine_name": "spacy",
              "models": [{"lang_code": k, "model_name": v} for k, v in SPACY_MODELS.items()]}
    nlp = NlpEngineProvider(nlp_configuration=config).create_engine()
    registry = RecognizerRegistry(supported_languages=list(SPACY_MODELS))
    registry.load_predefined_recognizers(languages=list(SPACY_MODELS), nlp_engine=nlp)
    return AnalyzerEngine(nlp_engine=nlp, registry=registry, supported_languages=list(SPACY_MODELS))


def presidio_all(analyzer, text, lang):
    """Every finding of Presidio, at any score: (start, end, entity type, score)."""
    return [(r.start, r.end, r.entity_type, r.score) for r in analyzer.analyze(text, language=lang)]


def presidio_defaults(findings):
    """Presidio as it comes: every entity type, every score."""
    return [(s, e, t) for s, e, t, _ in findings]


def presidio_tuned(findings, threshold=0.4):
    """Presidio set for this policy: no dates, organisations or URLs, and a minimum score."""
    return [(s, e, t) for s, e, t, score in findings if t not in NOT_PERSONAL and score >= threshold]

import time
analyzer = make_presidio()
start = time.time()
found = {d.doc_id: presidio_all(analyzer, d.text, d.lang) for d in docs.itertuples()}
print(f"Presidio ran on {len(found)} documents in {time.time() - start:.0f} s")

**(c) The model** is `chat-small`, asked to list every personal value as JSON. The next cell replays its recorded answers on the test part and turns each listed value into spans. A value that the model did not copy exactly is not in the text, so it masks nothing.

> **Check.** You should see 900 answers and the number of listed values that are not in the document.

In [ ]:
# Detector (c): a language model asked to list the personal data. The notebook replays recorded answers.
MODEL_TYPES = ["name", "email", "phone", "address", "government_id", "customer_or_staff_id", "bank_account",
               "card", "secret", "online_id", "date_of_birth"]
DETECT_PROMPT = (
    "You find personal data in documents before they are sent to another system. "
    "List every piece of personal data in the document: names of people, email addresses, phone numbers, "
    "street addresses and coordinates, government ID numbers (social security, passport, driving licence), "
    "customer and employee IDs, bank account numbers (IBAN, BBAN, routing numbers), card numbers and security "
    "codes, passwords, PINs and API keys, IP addresses and user names, and dates of birth. "
    "Do not list company names, ordinary dates and times, amounts or bank codes (SWIFT/BIC). "
    "Copy each value exactly as it appears in the document, character for character. "
    "The document can be in English, French or German.")
DETECT_SCHEMA = {
    "type": "object", "additionalProperties": False, "required": ["items"],
    "properties": {"items": {"type": "array", "items": {
        "type": "object", "additionalProperties": False, "required": ["text", "type"],
        "properties": {"text": {"type": "string"}, "type": {"type": "string", "enum": MODEL_TYPES}}}}}}


def detect_request(text):
    """The request body for one document (without the model name)."""
    return {"messages": [{"role": "system", "content": DETECT_PROMPT},
                         {"role": "user", "content": f"Document:\n<<<\n{text}\n>>>"}],
            "response_format": {"type": "json_schema",
                                "json_schema": {"name": "personal_data", "strict": True, "schema": DETECT_SCHEMA}}}


def spans_from_items(text, items):
    """Turn the model's list of values into spans: every place where each value occurs in the text.
    A value that is not in the text (the model changed it) gives no span: that is a miss."""
    spans = []
    for item in items:
        value = item["text"].strip()
        if len(value) < 2:
            continue
        for m in re.finditer(re.escape(value), text):
            spans.append((m.start(), m.end(), item["type"].upper()))
    return spans

class Replay:
    """Answers each model request from the recordings: the same document and task give the recorded answer.
    A request that differs from the recorded one is noted; a missing one raises an error."""

    def __init__(self, rows):
        self.rows = {(r["task"], r["doc_id"], r.get("condition", "")): r for r in rows}
        self.notes = []

    def answer(self, task, doc_id, request, condition=""):
        r = self.rows.get((task, doc_id, condition))
        if r is None:
            raise KeyError(f"no recording for {task} {doc_id} {condition}")
        if r["request_sha"] != request_sha(request):
            self.notes.append((task, doc_id, condition))
        return r


def request_sha(request):
    import hashlib
    return hashlib.sha256(json.dumps(request, sort_keys=True, ensure_ascii=False).encode()).hexdigest()[:16]

def unhide(r, text):
    """The labs copy hides the made-up keys in the answers (for example <FAKE-AWS-KEY-1>), so that secret
    scanners do not flag them. Put each key back from the document: "hidden" says where it is."""
    content = r.get("content") or ""
    for placeholder, (start, end) in r.get("hidden", {}).items():
        content = content.replace(placeholder, text[start:end])
    return content

replay = Replay(recordings["detect"])
model_spans, not_in_text = {}, []
texts = dict(zip(docs.doc_id, docs.text))
for d in test.itertuples():
    r = replay.answer("detect", d.doc_id, detect_request(d.text))
    content = unhide(r, d.text)
    items = json.loads(content)["items"] if content else []
    not_in_text += [i["text"] for i in items if i["text"].strip() and i["text"].strip() not in d.text]
    model_spans[d.doc_id] = spans_from_items(d.text, items)
print(len(model_spans), "answers |", len(not_in_text), "listed values not in the document | request notes:", len(replay.notes))
print(not_in_text[:8])

## 6. Change one thing on the development part: Presidio's settings

Presidio as it comes masks dates and company names too. The next cell compares the two settings on the development part, by language.

> **Predict.** Which setting hides more personal data? Which one masks more of the text that the model needs?

In [ ]:
def table(designs, part_docs, part_gold):
    """One row per design: values, the share hidden, partly hidden and missed, and the over-masking."""
    rows = {}
    for name, spans in designs.items():
        s = summary(part_docs, part_gold, spans)
        rows[name] = {"values": s["values"], "hidden": pct(s["hidden"]), "partly": pct(s["partly"]),
                      "missed": pct(s["missed"]), "over-masked": pct(s["over_masked"])}
    return pd.DataFrame(rows).T

designs = {"rules": {k: detect_rules(t) for k, t in zip(docs.doc_id, docs.text)},
           "presidio_defaults": {k: presidio_defaults(v) for k, v in found.items()},
           "presidio_tuned": {k: presidio_tuned(v) for k, v in found.items()}}
designs["rules_presidio"] = {k: designs["rules"][k] + designs["presidio_tuned"][k] for k in docs.doc_id}
gold_dev = gold[gold.split == "dev"]
table(designs, dev, gold_dev)

> **Check.** The defaults hide more personal values, because they mask every date and organisation, and with them some dates of birth and ID numbers. They pay for it with much more over-masking. The tuned setting masks far less of the useful text. Rules plus the tuned Presidio hide the most of the local designs, with little over-masking. That is the design this case study chooses **on the development part**, before it looks at the test.

> **Your turn.** Change one thing: lower Presidio's minimum score from 0.4 to 0.3, and compute the share of personal values hidden on the development part. Use `presidio_tuned(findings, threshold=0.3)` and `summary(...)`. Put the share (a number between 0 and 1, from `summary`) in `hidden_at_03`.

In [ ]:
hidden_at_03 = ...  # your answer: a number between 0 and 1

In [ ]:
expect('the share hidden by Presidio with a minimum score of 0.3', hidden_at_03, np.float64(0.4995), tolerance=1e-3)

> **Check.** A lower minimum score adds very little: the findings that it lets in are mostly weak guesses of shapes, not the missed names. To find more names, you need a better name detector, not a lower threshold.

## 7. The fair comparison, once, on the test part

Every design now runs on the same 900 test documents, with the same labels and the same two measures. The model's design is scored only on the test part, because it was recorded only there.

> **Predict.** Rank the five designs by the share of personal values hidden.

In [ ]:
designs["model"] = model_spans
designs["rules_presidio_model"] = {k: designs["rules_presidio"][k] + model_spans.get(k, []) for k in test.doc_id}
gold_test = gold[gold.split == "test"]
table(designs, test, gold_test)

The next cell splits the share hidden by language, with 95% intervals, for each design.

In [ ]:
rows = []
for name, spans in designs.items():
    s = score_values(test, gold_test[gold_test.role == "mask"], spans)
    for lang in LANGUAGES:
        x = s[s.lang == lang]
        k = int((x.status == "hidden").sum())
        lo, hi = wilson(k, len(x))
        rows.append({"design": name, "language": LANGUAGES[lang], "hidden": f"{k}/{len(x)}", "share": pct(k / len(x)),
                     "95% interval": f"{pct(lo)} to {pct(hi)}"})
pd.DataFrame(rows).set_index(["design", "language"])

> **Check.** Compare with the charts on the lesson page. Read the model's row with care: its prompt and the data's languages matter, and the page explains what its French and German numbers mean.

## 8. Where the chosen design fails

The next cell shows, for the chosen design (rules plus the tuned Presidio), the share hidden by group and language. A cell with fewer than 20 values shows its count only.

In [ ]:
chosen = designs["rules_presidio"]
s = score_values(test, gold_test[gold_test.role == "mask"], chosen)
def cell(x):
    k, n = int((x.status == "hidden").sum()), len(x)
    return f"{pct(k / n)} ({n})" if n >= 20 else f"{k}/{n}"
s.groupby(["group", "lang"]).apply(cell).unstack().loc[GROUPS]

And one example of each failure. The next cell prints, by language, values that were missed or only partly hidden, and words masked that are not personal data.

In [ ]:
for lang in LANGUAGES:
    x = s[s.lang == lang]
    print(LANGUAGES[lang])
    for status in ("missed", "partly"):
        print(f"  {status}:", list(dict.fromkeys(x[(x.status == status) & (x.group.isin(["name", "address"]))].value))[:5])
over = []
for d in test[test.lang == "fr"].head(40).itertuples():
    personal = [(g.start, g.end) for g in gold_test[(gold_test.doc_id == d.doc_id) & gold_test.role.isin(["mask", "placeholder"])].itertuples()]
    over += [d.text[a:b] for a, b, _ in merge(chosen[d.doc_id]) if not any(a < e and st < b for st, e in personal)]
print("Masked in French documents, but not personal data (first 12):", over[:12])

> **Your turn.** Decide what still needs a person. The rule of this case study: a group-and-language cell with **at least 20 values** whose share hidden is **under 90%** cannot be trusted to automatic masking. Make a sorted list of `(group, language code)` pairs that meet the rule, for example `("name", "fr")`. Put it in `needs_review`.

In [ ]:
needs_review = ...  # your answer: a sorted list of (group, lang) pairs

In [ ]:
expect_hash('the cells that need a person', needs_review, '813cbcb556536621')

## 9. What the masking costs the model

Masking protects people, but the model reads less. Two recorded tests measure the cost, each with the same documents in three versions: the original text, the text masked by the chosen design (`masked`), and the text masked by Presidio's defaults (`masked_defaults`).

1. **Routing**: the model chooses the document's type from five choices.
2. **Facts**: the model lists the dates and company names in the document; the score is the share of the labelled dates and companies it gives back.

> **Predict.** Will masking hurt routing? Will it hurt the facts? Which masking will hurt more?

In [ ]:
# The cost of masking: can the model still tell what kind of document it is? Five choices: the true type
# and four other types from the sample, in a fixed random order.
DOCTYPE_PROMPT = ("You sort incoming documents for a bank's back office. Read the document and choose the type "
                  "that fits it best. Answer with the number of one choice. Some values in the document may be "
                  "replaced by markers such as <PERSON> or <EMAIL>.")
DOCTYPE_SCHEMA = {"type": "object", "additionalProperties": False, "required": ["choice"],
                  "properties": {"choice": {"type": "integer", "enum": [1, 2, 3, 4, 5]}}}


def doctype_options(doc_id, true_type, all_types):
    """The five choices for one document; the same every time."""
    import random
    rng = random.Random(f"doctype-{doc_id}")
    others = rng.sample(sorted(t for t in set(all_types) if t != true_type), 4)
    options = others + [true_type]
    rng.shuffle(options)
    return options


def doctype_request(text, options):
    listed = "\n".join(f"{i}. {o}" for i, o in enumerate(options, 1))
    return {"messages": [{"role": "system", "content": DOCTYPE_PROMPT},
                         {"role": "user", "content": f"Document:\n<<<\n{text}\n>>>\n\nChoices:\n{listed}"}],
            "response_format": {"type": "json_schema",
                                "json_schema": {"name": "document_type", "strict": True, "schema": DOCTYPE_SCHEMA}}}


FACTS_PROMPT = ("You read documents for a bank's back office. List every calendar date and every company or "
                "organisation name in the document. Copy each one exactly as it appears. Some values in the "
                "document may be replaced by markers such as <PERSON> or <DATE_TIME>; do not list markers.")
FACTS_SCHEMA = {"type": "object", "additionalProperties": False, "required": ["dates", "companies"],
                "properties": {"dates": {"type": "array", "items": {"type": "string"}},
                               "companies": {"type": "array", "items": {"type": "string"}}}}


def facts_request(text):
    return {"messages": [{"role": "system", "content": FACTS_PROMPT},
                         {"role": "user", "content": f"Document:\n<<<\n{text}\n>>>"}],
            "response_format": {"type": "json_schema",
                                "json_schema": {"name": "facts", "strict": True, "schema": FACTS_SCHEMA}}}


def norm(value):
    return re.sub(r"\W+", "", value).lower()


def facts_found(gold_values, answered):
    """How many of the document's labelled values the answer contains (letters and digits compared)."""
    said = {norm(a) for a in answered}
    return sum(norm(v) in said for v in gold_values), len(gold_values)

types = dict(zip(docs.doc_id, docs.document_type))
route = pd.DataFrame([{"condition": r["condition"], "lang": r["doc_id"][:2],
                       "right": r["options"][json.loads(r["content"])["choice"] - 1] == types[r["doc_id"]]}
                      for r in recordings["doctype"] if r.get("content")])
print("Routing: share of documents with the right type")
print(route.pivot_table(index="condition", columns="lang", values="right", aggfunc="mean").map(pct))

keep = gold_test[(gold_test.role == "keep") & gold_test.label.isin(["date", "company"])]
rows = []
for r in recordings["facts"]:
    answer = json.loads(r["content"]) if r.get("content") else {"dates": [], "companies": []}
    for label, key in (("date", "dates"), ("company", "companies")):
        values = list(keep[(keep.doc_id == r["doc_id"]) & (keep.label == label)].value)
        got, n = facts_found(values, answer[key])
        rows.append({"condition": r["condition"], "label": label, "found": got, "n": n})
facts = pd.DataFrame(rows).groupby(["condition", "label"])[["found", "n"]].sum()
facts["share"] = (facts.found / facts.n).map(pct)
facts

> **Check.** Routing barely changes: the type of a document does not depend on who it is about. The facts test shows the cost of **over-masking**: Presidio's defaults mask dates and organisations, so the model can no longer give them back. Masking only what the policy says keeps most of them.

## 10. End to end: one new document

A new letter arrives, in French. The next cell masks it with the chosen design and prints what the model would receive, and which values a person must still check.

> **Your turn.** Change the letter: add your own made-up name and a made-up IBAN, run the cell again, and see what is masked.

In [ ]:
letter = ("Objet : remboursement\n"
          "Madame, Monsieur,\n"
          "Je m'appelle Hélène Marchetti, cliente n° CL-4471982. Le 3 mars 2026, votre agence de Lyon a débité "
          "deux fois 129,90 € sur mon compte FR76 3000 6000 0112 3456 7890 189. Merci de me rembourser. "
          "Vous pouvez me joindre au 06 12 34 56 78 ou à helene.marchetti@exemple.fr.\n"
          "Adresse : 14 rue des Tanneurs, 69002 Lyon\n"
          "Cordialement, Hélène Marchetti")
lang = "fr"
spans = merge(detect_rules(letter) + presidio_tuned(presidio_all(analyzer, letter, lang)))
print(mask(letter, spans))
print()
print("Found:", [(letter[a:b], label) for a, b, label in spans])
groups_to_check = sorted({g for g, l in needs_review if l == lang}) if isinstance(needs_review, list) else []
print("A person checks, in this language:", groups_to_check or "(finish section 8 first)")

> **Check.** Read the masked letter as the model would. The name, the customer number, the IBAN, the phone, the email and the address are masked. The date and the amount stay: the model needs them to handle the refund. Now look for over-masking: the greeting `Madame, Monsieur`, the branch's city and the word `Cordialement` are masked too, although they identify nobody.

## Optional: a live run with your own key

Everything above used recordings. To ask a model of your own for the personal data in one document, set `LIVE = True` and the environment variables `LIVE_BASE_URL`, `LIVE_API_KEY` and `LIVE_MODEL` for an OpenAI-compatible endpoint. It costs money, and the document's text goes to that provider: use only this made-up letter, never real data. A different model gives different results.

In [ ]:
LIVE = False          # set to True to call a real model (costs money; the text goes to the provider)

if LIVE:
    from openai import OpenAI
    client = OpenAI(base_url=os.environ["LIVE_BASE_URL"], api_key=os.environ["LIVE_API_KEY"])
    resp = client.chat.completions.create(model=os.environ["LIVE_MODEL"], **detect_request(letter))
    items = json.loads(resp.choices[0].message.content)["items"]
    print(mask(letter, spans_from_items(letter, items)))
else:
    print("Live run is off. Everything above used the recordings.")

## Recap

- You wrote a **masking policy** first: what must not reach the model, and what the model needs.
- You checked the labels: placeholders, role words and many unlabelled emails. The measured misses are a lower bound.
- You compared rules, Presidio and a recorded model on one fixed test part, with two measures: values **hidden** and text **over-masked**.
- You found where masking fails: names and addresses, more in some languages than in others.
- You decided which cells need a person, and measured what the masking costs the model.

Read the lesson page for the numbers in context, the limits, responsible use and how to make this a portfolio piece.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Threat-model and test an open-source AI application](https://learning.nextia-ai.com/courses/ai-security/m07/threat-model-an-open-source-ai-app/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ai-security/m07/find-and-remove-personal-data/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The documents are Gretel's synthetic_pii_finance_multilingual (Apache-2.0); every person and number in them is made up. The model answers are real recordings.